In [1]:
import pandas as pd
data_train = pd.read_parquet(r"final_data\chunk-based-split\train_df_prepared.parquet")
data_valid = pd.read_parquet(r"final_data\chunk-based-split\valid_df_prepared.parquet")
data_test = pd.read_parquet(r"final_data\chunk-based-split\test_df_prepared.parquet")

In [2]:
import pandas as pd
import numpy as np
import torch
import gc
from torch_geometric.data import Data
from tqdm.notebook import tqdm
import os

# Ngăn chặn cảnh báo OpenMP
os.environ["KMP_DUPLICATE_BLOCK_WARNING"] = "TRUE"

# Hàm V3: Xây dựng đồ thị Temporal IP Topology CÓ ĐẶC TRƯNG CẠNH (Edge Attributes)
def build_ip_topology_graph(df, train_mask, valid_mask, test_mask, window_size=5):
    print(f"Bước 1: Trích xuất Node Features (X) và Label (y)...")
    
    cols_to_drop = ['flow_id', 'timestamp', 'timestamp_num', 'src_ip', 'dst_ip', 'protocol', 'src_port', 'dst_port', 'label']
    cols_present = [c for c in cols_to_drop if c in df.columns]
    feature_cols = [c for c in df.columns if c not in cols_present]
    
    features_np = np.empty((len(df), len(feature_cols)), dtype=np.float32)
    for i, col in enumerate(feature_cols):
        features_np[:, i] = df[col].astype(np.float32).values
            
    features_np = np.ascontiguousarray(features_np)
    
    x_tensor = torch.from_numpy(features_np).contiguous()
    y_tensor = torch.tensor(df['label'].values, dtype=torch.long).contiguous()

    print(f"Bước 2: Xây Dựng Đồ Thị Dây Chuyền Theo IP (Temporal IP Topology)...")
    timestamps = df['timestamp_num'].values
    src_ips = df['src_ip'].values
    dst_ips = df['dst_ip'].values
    src_ports = df['src_port'].astype(str).values
    dst_ports = df['dst_port'].astype(str).values
    
    # 0=Train, 1=Valid, 2=Test (Chống Data Leakage)
    split_id = np.zeros(len(df), dtype=np.int8)
    split_id[valid_mask.numpy()] = 1
    split_id[test_mask.numpy()] = 2
    
    ip_to_indices = {}
    for idx in tqdm(range(len(df)), desc="[1/2] Gom nhóm Gói tin theo IP"):
        s_ip = src_ips[idx]
        d_ip = dst_ips[idx]
        
        if s_ip not in ip_to_indices: ip_to_indices[s_ip] = []
        if len(ip_to_indices[s_ip]) == 0 or ip_to_indices[s_ip][-1] != idx:
            ip_to_indices[s_ip].append(idx)
            
        if d_ip not in ip_to_indices: ip_to_indices[d_ip] = []
        if len(ip_to_indices[d_ip]) == 0 or ip_to_indices[d_ip][-1] != idx:
            ip_to_indices[d_ip].append(idx)
            
    all_src, all_dst, all_edge_attrs = [], [], []
    
    for ip, indices in tqdm(ip_to_indices.items(), desc=f"[2/2] Căng lưới đồ thị (Window = {window_size})"):
        n_flows = len(indices)
        if n_flows < 2: continue
        
        for i in range(1, n_flows):
            curr_idx = indices[i]
            start_w = max(0, i - window_size)
            
            for j in range(start_w, i):
                past_idx = indices[j]
                
                dt_raw = abs(timestamps[curr_idx] - timestamps[past_idx])
                # [QUAN TRỌNG NHẤT] NGẮT SESSION (SESSION CUTOFF):
                # Nếu 2 gói tin cách nhau quá 60 giây (khác ngày, khác đợt tấn công),
                # thì TUYỆT ĐỐI KHÔNG nối cạnh để tránh GAT trộn lẫn nhiễu (muddying).
                if dt_raw > 60.0:
                    continue
                
                # -- TÍNH TOÁN 4 ĐẶC TRƯNG CẠNH (EDGE ATTRIBUTES) --
                # 1. Delta Time (Log scale + Min-Max Scaling về dải [0, 1])
                # Giá trị max lý thuyết khi dt_raw = 60s là log1p(60,000,000) ~ 17.9
                dt = np.log1p(dt_raw * 1e6) / 18.0
                # 2. Same Direction (Cùng hướng IP)
                same_dir = 1.0 if src_ips[curr_idx] == src_ips[past_idx] else 0.0
                # 3. Same Src Port (Kiểm tra Botnet/Flood randomizaton)
                same_sport = 1.0 if src_ports[curr_idx] == src_ports[past_idx] else 0.0
                # 4. Same Dst Port (Kiểm tra mục tiêu chọc vào tường lửa)
                same_dport = 1.0 if dst_ports[curr_idx] == dst_ports[past_idx] else 0.0
                
                attr = [dt, same_dir, same_sport, same_dport]
                
                # QUY TẮC STRICT INDUCTIVE TOÁN HỌC:
                if split_id[curr_idx] == 0 and split_id[past_idx] == 0:
                    # Nối 2 chiều trên tập Train
                    all_src.extend([curr_idx, past_idx])
                    all_dst.extend([past_idx, curr_idx])
                    all_edge_attrs.extend([attr, attr]) 
                else:
                    # Nối 1 chiều về mặt thời gian cho tập Test/Valid
                    all_src.append(past_idx)
                    all_dst.append(curr_idx)
                    all_edge_attrs.append(attr) 
                    
    print("\nHợp nhất và tạo Cạnh (Culling Edge Index)...")
    # Sử dụng Pandas để loại bỏ các cạnh trùng lặp chính xác
    df_edges = pd.DataFrame({
        'src': all_src, 
        'dst': all_dst,
        'attr0': [a[0] for a in all_edge_attrs],
        'attr1': [a[1] for a in all_edge_attrs],
        'attr2': [a[2] for a in all_edge_attrs],
        'attr3': [a[3] for a in all_edge_attrs]
    })
    df_edges = df_edges.drop_duplicates(subset=['src', 'dst']).reset_index(drop=True)
    
    edge_index = torch.tensor(df_edges[['src', 'dst']].values.T, dtype=torch.long).contiguous()
    edge_attr = torch.tensor(df_edges[['attr0', 'attr1', 'attr2', 'attr3']].values, dtype=torch.float32).contiguous()
    
    print(f"HOÀN THIỆN ĐỒ THỊ TEMPORAL IP (CÓ EDGE FEATURES)!")
    print(f"   - Số Nodes : {x_tensor.size(0):,}")
    print(f"   - Số Edges : {edge_index.size(1):,}")
    print(f"   - Số Features/Cạnh : {edge_attr.shape[1]}")
    
    del features_np, df_edges, all_src, all_dst, all_edge_attrs
    gc.collect()
    
    graph = Data(x=x_tensor, edge_index=edge_index, edge_attr=edge_attr, y=y_tensor)
    graph.train_mask = train_mask
    graph.valid_mask = valid_mask
    graph.test_mask = test_mask
    return graph

In [3]:
import gc
from sklearn.preprocessing import LabelEncoder, QuantileTransformer
from torch_geometric.loader import NeighborLoader
import torch
import numpy as np
import pandas as pd

print("=== BƯỚC 1: TIỀN XỬ LÝ VÀ CHUẨN BỊ STRICT INDUCTIVE IP GRAPH ===")

print("Đánh dấu tập dữ liệu và gộp 3 DataFrames thành một Siêu Dataframe...")
data_train['split_tag'] = 0
data_valid['split_tag'] = 1
data_test['split_tag'] = 2

df_all = pd.concat([data_train, data_valid, data_test], ignore_index=True)

# Sửa lại thành phép chia thập phân (/) và ép kiểu datetime trước để giữ lại Micro giây
df_all['timestamp_num'] = pd.to_datetime(df_all['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

print("Sắp xếp toàn bộ gói tin lồng nhau theo trình tự thời gian (Chronological Sort)...")
df_all = df_all.sort_values(by='timestamp_num').reset_index(drop=True)

total_nodes = len(df_all)

print("Tái tạo Cấu trúc Masks động (Dynamic Masking) bảo tồn tính chặt chẽ sau khi Sort...")
train_mask = torch.tensor((df_all['split_tag'] == 0).values, dtype=torch.bool)
valid_mask = torch.tensor((df_all['split_tag'] == 1).values, dtype=torch.bool)
test_mask  = torch.tensor((df_all['split_tag'] == 2).values, dtype=torch.bool)

# Xóa cột đánh dấu để không thành features nhiễu
df_all = df_all.drop(columns=['split_tag'])

print(f" - Tổng Node: {total_nodes:,}")
print(f" - Train Mask : {train_mask.sum().item():,} ({train_mask.float().mean()*100:.1f}%)")
print(f" - Valid Mask : {valid_mask.sum().item():,} ({valid_mask.float().mean()*100:.1f}%)")
print(f" - Test Mask  : {test_mask.sum().item():,} ({test_mask.float().mean()*100:.1f}%)")

print("Hoàn tất Data Prep cho Masks và Sorting Thời gian.")

print("\n=== BẮT ĐẦU QUÁ TRÌNH BUILD IP TOPOLOGY INDEX ===")
super_graph_faiss = build_ip_topology_graph(df_all, train_mask, valid_mask, test_mask, window_size=10)
print("\nĐồ thị tổng IP Topo Đã Gắn Mask Không Rò Rỉ Thành Công!")

# 6. Set up loaders chống OOM VRAM
print("\nSet up IP Graph NeighborLoaders...")
train_loader_faiss = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.train_mask, batch_size=2048, shuffle=True, num_workers=0)
valid_loader_faiss = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.valid_mask, batch_size=2048, shuffle=False, num_workers=0)
test_loader_faiss  = NeighborLoader(super_graph_faiss, num_neighbors=[10, 5], input_nodes=super_graph_faiss.test_mask,  batch_size=2048, shuffle=False, num_workers=0)

del df_all 
gc.collect()
print("Hoàn tất Data Prep cho IP Graph (Hoàn toàn không có Data Leakage)!")

=== BƯỚC 1: TIỀN XỬ LÝ VÀ CHUẨN BỊ STRICT INDUCTIVE IP GRAPH ===
Đánh dấu tập dữ liệu và gộp 3 DataFrames thành một Siêu Dataframe...
Sắp xếp toàn bộ gói tin lồng nhau theo trình tự thời gian (Chronological Sort)...
Tái tạo Cấu trúc Masks động (Dynamic Masking) bảo tồn tính chặt chẽ sau khi Sort...
 - Tổng Node: 3,801,012
 - Train Mask : 2,470,607 (65.0%)
 - Valid Mask : 570,114 (15.0%)
 - Test Mask  : 760,291 (20.0%)
Hoàn tất Data Prep cho Masks và Sorting Thời gian.

=== BẮT ĐẦU QUÁ TRÌNH BUILD IP TOPOLOGY INDEX ===
Bước 1: Trích xuất Node Features (X) và Label (y)...
Bước 2: Xây Dựng Đồ Thị Dây Chuyền Theo IP (Temporal IP Topology)...


[1/2] Gom nhóm Gói tin theo IP:   0%|          | 0/3801012 [00:00<?, ?it/s]

[2/2] Căng lưới đồ thị (Window = 10):   0%|          | 0/23 [00:00<?, ?it/s]


Hợp nhất và tạo Cạnh (Culling Edge Index)...
HOÀN THIỆN ĐỒ THỊ TEMPORAL IP (CÓ EDGE FEATURES)!
   - Số Nodes : 3,801,012
   - Số Edges : 107,911,886
   - Số Features/Cạnh : 4

Đồ thị tổng IP Topo Đã Gắn Mask Không Rò Rỉ Thành Công!

Set up IP Graph NeighborLoaders...
Hoàn tất Data Prep cho IP Graph (Hoàn toàn không có Data Leakage)!


In [4]:
import torch
import torch.nn.functional as F
from torch_geometric.loader import NeighborLoader
from torch_geometric.nn import GATv2Conv
from torch_geometric.utils import dropout_edge
import torch.nn as nn
from tqdm.notebook import tqdm
import copy
import gc

def create_dataloader(graph_data, batch_size=2048, shuffle=True):
    print(f"Đang khởi tạo NeighborLoader (Kích thước Batch: {batch_size}, Shuffle: {shuffle})...")
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[5, 3], 
        batch_size=batch_size,
        shuffle=shuffle, 
        num_workers=0 
    )
    return loader

class GAT_Embedder(torch.nn.Module):
    # Thêm nhận diện edge_dim = 4 để GAT biết cách ghim Attention Weight vào cạnh
    def __init__(self, in_channels, hidden_channels, embedding_dim, num_classes, heads=4, edge_dropout=0.1, edge_dim=4):
        super(GAT_Embedder, self).__init__()
        self.edge_dropout = edge_dropout 
        
        # Bổ sung edge_dim vào các lớp Conv. Đã bỏ lớp skip, cấu trúc giống gat_training.ipynb
        self.conv1 = GATv2Conv(in_channels, hidden_channels, heads=heads, dropout=0.1, edge_dim=edge_dim)
        self.bn1 = nn.BatchNorm1d(hidden_channels * heads)
        
        self.conv2 = GATv2Conv(hidden_channels * heads, embedding_dim, heads=1, concat=False, dropout=0.1, edge_dim=edge_dim)
        self.bn2 = nn.BatchNorm1d(embedding_dim)
        
        self.classifier = nn.Linear(embedding_dim, num_classes)

    def forward(self, x, edge_index, edge_attr):
        # GRAPH CONV 1
        edge_index_dp, edge_mask = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp = edge_attr[edge_mask] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.1, training=self.training)
        x = self.conv1(x, edge_index_dp, edge_attr=edge_attr_dp)
        x = self.bn1(x)
        x = F.elu(x)
        
        # GRAPH CONV 2
        edge_index_dp2, edge_mask2 = dropout_edge(edge_index, p=self.edge_dropout, force_undirected=False, training=self.training)
        edge_attr_dp2 = edge_attr[edge_mask2] if edge_attr is not None else None
        
        x = F.dropout(x, p=0.1, training=self.training)
        embedding = self.conv2(x, edge_index_dp2, edge_attr=edge_attr_dp2)
        embedding = self.bn2(embedding)
        embedding = F.elu(embedding) 
        
        out = self.classifier(embedding)
        return out, embedding

def train_gat(model, train_loader, valid_loader, device, epochs=5, lr=0.005, class_weights=None, patience=7):
    from sklearn.metrics import f1_score
    import numpy as np

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-3) 
    
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=3, verbose=True
    )
    
    if class_weights is not None:
        criterion = nn.CrossEntropyLoss(weight=class_weights)
    else:
        criterion = nn.CrossEntropyLoss()
        
    best_val_loss = float('inf')
    best_model_weights = None
    no_improve_epochs = 0
    
    for epoch in range(epochs):
        model.train()
        total_train_loss = 0
        total_train_nodes = 0
        all_train_preds = []
        all_train_labels = []
        
        pbar_train = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for batch in pbar_train:
            batch = batch.to(device)
            optimizer.zero_grad()
            
            # TRUYỀN EDGE_ATTR VÀO MODEL
            out, _ = model(batch.x, batch.edge_index, batch.edge_attr)
            batch_size = batch.batch_size
            
            loss = criterion(out[:batch_size], batch.y[:batch_size])
            loss.backward()
            
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
            
            optimizer.step()
            
            pred = out[:batch_size].argmax(dim=1)
            y_true = batch.y[:batch_size]
            
            all_train_preds.append(pred.cpu().numpy())
            all_train_labels.append(y_true.cpu().numpy())
            
            total_train_loss += loss.item() * batch_size
            total_train_nodes += batch_size
            
            pbar_train.set_postfix({'Loss': f"{loss.item():.4f}"})
            
            # Giải phóng Tensor rác mỗi batch
            del batch, out, loss, pred, y_true
            
        avg_train_loss = total_train_loss / total_train_nodes
        train_f1 = f1_score(np.concatenate(all_train_labels), np.concatenate(all_train_preds), average='macro')
        
        # Đóng gọn pbar. KHÔNG GỌI empty_cache() và gc.collect() ở đây
        pbar_train.close()
            
        model.eval()
        total_val_loss = 0
        total_val_nodes = 0
        all_val_preds = []
        all_val_labels = []
        
        with torch.no_grad():
            pbar_val = tqdm(valid_loader, desc=f"Epoch {epoch+1}/{epochs} [Valid]")
            for batch in pbar_val:
                batch = batch.to(device)
                
                # TRUYỀN EDGE_ATTR VÀO MODEL KHI VALIDATE
                out, _ = model(batch.x, batch.edge_index, batch.edge_attr)
                batch_size = batch.batch_size
                
                loss = criterion(out[:batch_size], batch.y[:batch_size])
                pred = out[:batch_size].argmax(dim=1)
                y_true = batch.y[:batch_size]
                
                all_val_preds.append(pred.cpu().numpy())
                all_val_labels.append(y_true.cpu().numpy())
                
                total_val_loss += loss.item() * batch_size
                total_val_nodes += batch_size
                
                pbar_val.set_postfix({'Loss': f"{loss.item():.4f}"})
                
                # Giải phóng Tensor rác mỗi batch
                del batch, out, loss, pred, y_true
                
        # Đóng gọn pbar. KHÔNG GỌI empty_cache() ở đây
        pbar_val.close()
                
        avg_val_loss = total_val_loss / total_val_nodes
        val_f1 = f1_score(np.concatenate(all_val_labels), np.concatenate(all_val_preds), average='macro')
        
        scheduler.step(avg_val_loss)
        
        print(f"=== Epoch {epoch+1} Tổng kết ===")
        print(f"   Train Loss: {avg_train_loss:.4f} | Train Macro F1: {train_f1:.4f}")
        print(f"   Valid Loss: {avg_val_loss:.4f}   | Valid Macro F1: {val_f1:.4f}")
        current_lr = optimizer.param_groups[0]['lr']
        print(f"   Learning Rate hiện tại: {current_lr:.6f}")
        
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            no_improve_epochs = 0
            best_model_weights = copy.deepcopy(model.state_dict())
            print(f"Best model. Đã lưu trọng số tốt nhất tại Epoch {epoch+1} (Val Loss: {best_val_loss:.4f})")
        else:
            no_improve_epochs += 1
            print(f"Báo động Early Stopping: Không cải thiện {no_improve_epochs}/{patience} epochs.")
            if no_improve_epochs >= patience:
                print(f"\n🛑 Đã kích hoạt Early Stopping tại Epoch {epoch+1}! Lùi về checkpoint tốt nhất.")
                break

    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        print(f"\nHoàn tất Train! Đã load lại weights tốt nhất (Val Loss: {best_val_loss:.4f}) để dùng trích xuất Embedding!")
        
    return model

In [5]:
import xgboost as xgb
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm.notebook import tqdm
import numpy as np
import torch
import gc
from torch_geometric.loader import NeighborLoader

@torch.no_grad()
def extract_embeddings_mask(model, graph_data, mask, device='cpu'):
    model.eval()
    print("Đang khởi tạo Inference Loader múc lấy Embeddings...")
    
    # [QUAN TRỌNG]: Loader dành riêng cho lúc test/extract (batch bự, ko shuffle, ko lấy hàng xóm rườm rà)
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[5, 3], # Lấy ít neighbor thôi
        input_nodes=mask,
        batch_size=16384,     # Kéo max Batch Size lên để GPU nhai nhanh
        shuffle=False,        # TUYỆT ĐỐI FALSE
        num_workers=0 
    )
    
    all_embeddings = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Đang trích xuất Vectơ Embedding")
    for batch in pbar:
        batch = batch.to(device)
        
        # Đẩy qua GAT, truyền kèm theo edge_attr
        _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
        
        # Chỉ tập trung thu Vector Embedding của các Core Nodes
        bs = batch.batch_size
        all_embeddings.append(emb[:bs].cpu().numpy())
        all_labels.append(batch.y[:bs].cpu().numpy())
        
        # [QUAN TRỌNG] Dọn rác Tensor Inference để tránh nghẽn VRAM và RAM SWAP
        del batch, emb, _
        
    pbar.close()
    
    # Tự động gom rác cho luồng Python
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        
    final_embeddings = np.concatenate(all_embeddings, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    print(f"Trích xuất thành công! Kích thước Ma trận Embeddings: {final_embeddings.shape}")
    return final_embeddings, final_labels

def train_evaluate_xgboost(X_train_emb, y_train, X_valid_emb, y_valid, X_test_emb, y_test):
    print("\n--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---")

    from sklearn.utils.class_weight import compute_class_weight
    print("Đang tính toán Custom Smoothed Class Weights...")
    
    unique_classes = np.unique(y_train)
    raw_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
    weights_dict = {c: np.sqrt(w) for c, w in zip(unique_classes, raw_class_weights)}
    
    # TINH CHỈNH THỦ CÔNG DỰA TRÊN KẾT QUẢ CLASSIFICATION REPORT GẦN ĐÂY:
    if 4 in weights_dict: weights_dict[4] *= 0.6
    if 6 in weights_dict: weights_dict[6] *= 1.5
    if 9 in weights_dict: weights_dict[9] *= 2.5
    if 1 in weights_dict: weights_dict[1] *= 0.8

    # Chuyển đổi thành mảng Sample Weight cho tập train
    final_sample_weights = np.array([weights_dict[y] for y in y_train])

    xgb_model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.08,
        max_depth=7, # [QUAN TRỌNG]: Tụt max_depth về 7 để tránh OOM GPU 
        min_child_weight=2,
        gamma=0.4,
        reg_lambda=2.0,
        reg_alpha=0.5,
        subsample=0.8,         
        colsample_bytree=0.9,
        max_delta_step=3,
        random_state=42,
        eval_metric='mlogloss',
        tree_method="hist",
        device="cuda",  
        early_stopping_rounds=40
    )
    
    print("🧹 Đang dọn dẹp RAM & VRAM (Garbage Collection)...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("⏳ Đang huấn luyện XGBoost... (Có sử dụng Valid Set để Tự động Dừng - Early Stopping)")
    # Wrap model fit logic
    try:
        xgb_model.fit(
            X_train_emb, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train_emb, y_train), (X_valid_emb, y_valid)],
            verbose=100 
        )
    except Exception as e:
        print(f"Lỗi XGBoost, thử fallback về CPU... Lỗi: {e}")
        xgb_model.set_params(device='cpu')
        xgb_model.fit(
            X_train_emb, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train_emb, y_train), (X_valid_emb, y_valid)],
            verbose=100 
        )
        
    print(f"Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: {xgb_model.best_iteration}")
    
    print("\n--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK % ---")
    y_pred = xgb_model.predict(X_test_emb)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    return xgb_model

In [6]:
import numpy as np
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from tqdm.notebook import tqdm
import torch
import gc
from torch_geometric.loader import NeighborLoader

@torch.no_grad()
def extract_concat_features_mask(model, graph_data, mask, device='cpu'):
    model.eval()
    print("Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...")
    
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[5, 3],
        input_nodes=mask,
        batch_size=16384,
        shuffle=False,
        num_workers=0 
    )
    
    all_combined = []
    all_labels = []
    
    pbar = tqdm(loader, desc="Đang trích xuất Vectơ Nối")
    for batch in pbar:
        batch = batch.to(device)
        
        # Đẩy qua GAT, truyền kèm edge_attr
        _, emb = model(batch.x, batch.edge_index, batch.edge_attr)
        
        bs = batch.batch_size
        
        # Lấy raw features và embeddings của các core node trong batch
        raw_features = batch.x[:bs].cpu().numpy()
        embeddings = emb[:bs].cpu().numpy()
        
        # Nối (concatenation) theo axis=1
        combined_matrix = np.concatenate([raw_features, embeddings], axis=1)
        
        all_combined.append(combined_matrix)
        all_labels.append(batch.y[:bs].cpu().numpy())
        
        # [QUAN TRỌNG] Giải phóng bộ nhớ tạm ngầm để mảng list dồn lại không bị đẩy RAM xuống SWAP
        del batch, emb, _, raw_features, embeddings, combined_matrix
        
    pbar.close()
    
    # Cuối tiến trình trích xuất dọn một lần duy nhất
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    
    final_combined = np.concatenate(all_combined, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)
    
    print(f"Trích xuất thành công! Kích thước Ma trận Nối: {final_combined.shape}")
    return final_combined, final_labels

def train_evaluate_concat_xgboost(X_train, y_train, X_valid, y_valid, X_test, y_test):
    print("\n--- BẮT ĐẦU TRAIN XGBOOST TỪ FEATURE + EMBEDDING (CONCAT) ---")

    from sklearn.utils.class_weight import compute_class_weight
    print("Đang tính toán Custom Smoothed Class Weights...")
    
    unique_classes = np.unique(y_train)
    raw_class_weights = compute_class_weight('balanced', classes=unique_classes, y=y_train)
    weights_dict = {c: np.sqrt(w) for c, w in zip(unique_classes, raw_class_weights)}
    
    if 4 in weights_dict: weights_dict[4] *= 0.6
    if 6 in weights_dict: weights_dict[6] *= 1.5
    if 9 in weights_dict: weights_dict[9] *= 2.5
    if 1 in weights_dict: weights_dict[1] *= 0.8

    final_sample_weights = np.array([weights_dict[y] for y in y_train])

    xgb_model = xgb.XGBClassifier(
        n_estimators=1000,
        learning_rate=0.08,
        max_depth=7,  # Giảm xuống 7 để tránh OOM
        min_child_weight=2,
        gamma=0.4,
        reg_lambda=2.0,
        reg_alpha=0.5,
        subsample=0.8,         
        colsample_bytree=0.9,
        max_delta_step=3,
        random_state=42,
        eval_metric='mlogloss',
        tree_method="hist",
        device="cuda",
        early_stopping_rounds=40
    )
    
    print("🧹 Đang dọn dẹp RAM & VRAM (Garbage Collection)...")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print("⏳ Đang huấn luyện XGBoost... (Early Stopping trên Valid Set)")
    try:
        xgb_model.fit(
            X_train, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train, y_train), (X_valid, y_valid)],
            verbose=100 
        )
    except Exception as e:
        print(f"Lỗi XGBoost, thử fallback về CPU... Lỗi: {e}")
        xgb_model.set_params(device='cpu')
        xgb_model.fit(
            X_train, y_train,
            sample_weight=final_sample_weights,
            eval_set=[(X_train, y_train), (X_valid, y_valid)],
            verbose=100 
        )
        
    print(f"Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: {xgb_model.best_iteration}")
    
    print("\n--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK ---")
    y_pred = xgb_model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')
    
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    
    print("Classification Report:")
    print(classification_report(y_test, y_pred, digits=4))
    
    return xgb_model

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Đang Train trên:", device)

print("=== BƯỚC 2: CHUẨN BỊ MÔ HÌNH VÀ HUẤN LUYỆN GAT TRÊN FAISS GRAPH ===")
gc.collect()

num_features = super_graph_faiss.x.shape[1] 
# PyG tự nhận diện số lượng class duy nhất trong label
num_classes = len(torch.unique(super_graph_faiss.y)) 
print(f"Số lượng Classes (Loại tấn công): {num_classes}")

# Khởi tạo mô hình GAT_Embedder đã được tinh chỉnh v2
model_faiss = GAT_Embedder(
     in_channels=num_features, 
     hidden_channels=128,      
     embedding_dim=64,        
     num_classes=num_classes, 
     heads=8,
     # TĂNG MẠNH EDGE DROPOUT LÊN 0.3: Không cho GAT tin tưởng hoàn toàn vào Đồ thị FAISS, 
     # ép model phải xài Data gốc qua Skip-Connections
     edge_dropout=0.3,
     edge_dim=4   # 4 Đặc trưng thời gian/IP/Cổn cho Cạnh
).to(device)

# Tạo mảng Class_Weights để phạt mạnh class thiểu số chống Overfitting XGB
import numpy as np
from sklearn.utils.class_weight import compute_class_weight

train_y_cpu = super_graph_faiss.y[super_graph_faiss.train_mask].cpu().numpy()
unique_classes_arr = np.unique(train_y_cpu)
raw_weights_arr = compute_class_weight('balanced', classes=unique_classes_arr, y=train_y_cpu)

# [QUAN TRỌNG]: LÀM MƯỢT TRỌNG SỐ LỚP CHỐNG SỐC LOSS (SMOOTHING)
smoothed_weights = np.sqrt(raw_weights_arr)
# Khóa trần (Clip) ở mức 10.0 để tránh gradient xé rách đạo hàm
smoothed_weights = np.clip(smoothed_weights, a_min=0.1, a_max=10.0)

gat_class_weights_faiss = torch.tensor(smoothed_weights, dtype=torch.float32).to(device)
print(f"Trọng số cân bằng Class Weights GAT (Sau khi vát mượt): {gat_class_weights_faiss}")

print("\nBắt đầu quá trình Huấn Luyện GAT (Phiên bản FAISS)!")
# Dùng hàm train_gat 
model_faiss = train_gat(
    model_faiss, 
    train_loader_faiss, 
    valid_loader_faiss, 
    device, 
    epochs=5, 
    lr=0.001, 
    class_weights=gat_class_weights_faiss,
    patience=2
)

Đang Train trên: cuda
=== BƯỚC 2: CHUẨN BỊ MÔ HÌNH VÀ HUẤN LUYỆN GAT TRÊN FAISS GRAPH ===
Số lượng Classes (Loại tấn công): 11
Trọng số cân bằng Class Weights GAT (Sau khi vát mượt): tensor([1.8663, 0.3778, 5.2457, 1.3807, 1.9085, 5.3146, 2.4158, 0.8020, 2.0317,
        1.2901, 1.9296], device='cuda:0')

Bắt đầu quá trình Huấn Luyện GAT (Phiên bản FAISS)!


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\torch\optim\lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1/5 [Train]:   0%|          | 0/1207 [00:00<?, ?it/s]

Epoch 1/5 [Valid]:   0%|          | 0/279 [00:00<?, ?it/s]

=== Epoch 1 Tổng kết ===
   Train Loss: 0.0861 | Train Macro F1: 0.9530
   Valid Loss: 0.0279   | Valid Macro F1: 0.9806
   Learning Rate hiện tại: 0.001000
Best model. Đã lưu trọng số tốt nhất tại Epoch 1 (Val Loss: 0.0279)


...

Epoch 2/5 [Valid]:   0%|          | 0/279 [00:00<?, ?it/s]

=== Epoch 2 Tổng kết ===
   Train Loss: 0.0414 | Train Macro F1: 0.9773
   Valid Loss: 0.0385   | Valid Macro F1: 0.9638
   Learning Rate hiện tại: 0.001000
Báo động Early Stopping: Không cải thiện 1/2 epochs.


Epoch 3/5 [Train]:   0%|          | 0/1207 [00:00<?, ?it/s]

Epoch 3/5 [Valid]:   0%|          | 0/279 [00:00<?, ?it/s]

=== Epoch 3 Tổng kết ===
   Train Loss: 0.0365 | Train Macro F1: 0.9806
   Valid Loss: 0.0771   | Valid Macro F1: 0.9693
   Learning Rate hiện tại: 0.001000
Báo động Early Stopping: Không cải thiện 2/2 epochs.

🛑 Đã kích hoạt Early Stopping tại Epoch 3! Lùi về checkpoint tốt nhất.

Hoàn tất Train! Đã load lại weights tốt nhất (Val Loss: 0.0279) để dùng trích xuất Embedding!


In [8]:
# giải phóng RAM
import gc
gc.collect()

print("=== BƯỚC 3: TRÍCH XUẤT EMBEDDINGS VÀ HUẤN LUYỆN HYBRID XGBOOST (FAISS MODEL) ===")
# Trích xuất sang Vector cho XGBoost xử lý
print("\n[Trích xuất Vector cho Train Mask FAISS]")
X_train_faiss, y_train_faiss = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.train_mask, device=device)

print("\n[Trích xuất Vector cho Valid Mask FAISS] - Vai trò Giám khảo Early Stopping")
X_valid_faiss, y_valid_faiss = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.valid_mask, device=device)

print("\n[Trích xuất Vector cho Test Mask FAISS]")
X_test_faiss, y_test_faiss = extract_embeddings_mask(model_faiss, super_graph_faiss, super_graph_faiss.test_mask, device=device)

# Huấn luyện và Đánh giá (Sử dụng hàm train_evaluate_xgboost cũ)
hybrid_xgb_faiss = train_evaluate_xgboost(X_train_faiss, y_train_faiss, X_valid_faiss, y_valid_faiss, X_test_faiss, y_test_faiss)

# Lưu Model cuối cùng
hybrid_xgb_faiss.save_model("GAT_XGB_Hybrid_FAISS_Model.json")
print("\n💾 Thành công! Hybrid XGBoost FAISS Model đã lưu vào 'GAT_XGB_Hybrid_FAISS_Model.json'")

=== BƯỚC 3: TRÍCH XUẤT EMBEDDINGS VÀ HUẤN LUYỆN HYBRID XGBOOST (FAISS MODEL) ===

[Trích xuất Vector cho Train Mask FAISS]
Đang khởi tạo Inference Loader múc lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/151 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (2470607, 64)

[Trích xuất Vector cho Valid Mask FAISS] - Vai trò Giám khảo Early Stopping
Đang khởi tạo Inference Loader múc lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/35 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (570114, 64)

[Trích xuất Vector cho Test Mask FAISS]
Đang khởi tạo Inference Loader múc lấy Embeddings...


Đang trích xuất Vectơ Embedding:   0%|          | 0/47 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Embeddings: (760291, 64)

--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---
Đang tính toán Custom Smoothed Class Weights...
🧹 Đang dọn dẹp RAM & VRAM (Garbage Collection)...
⏳ Đang huấn luyện XGBoost... (Có sử dụng Valid Set để Tự động Dừng - Early Stopping)
[0]	validation_0-mlogloss:1.56142	validation_1-mlogloss:1.56242
[100]	validation_0-mlogloss:0.00182	validation_1-mlogloss:0.02043
[126]	validation_0-mlogloss:0.00112	validation_1-mlogloss:0.02193
Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: 86

--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK % ---


c:\Users\Admin\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:751: UserWarning: [12:20:56] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


Accuracy:            99.12%
F1-Score (Macro):    97.01%
F1-Score (Weighted): 99.12%

Classification Report:
              precision    recall  f1-score   support

           0     0.9592    0.9924    0.9755     19851
           1     0.9989    1.0000    0.9994    484073
           2     0.9984    0.9925    0.9954      2521
           3     0.9997    0.9963    0.9980     36260
           4     0.9562    0.9080    0.9315     18981
           5     1.0000    0.9947    0.9974      2460
           6     0.8489    0.9357    0.8902     11851
           7     1.0000    0.9928    0.9964    107440
           8     0.9046    0.9861    0.9436     16760
           9     0.9992    0.9309    0.9638     41522
          10     0.9604    0.9995    0.9795     18572

    accuracy                         0.9912    760291
   macro avg     0.9659    0.9754    0.9701    760291
weighted avg     0.9916    0.9912    0.9912    760291


💾 Thành công! Hybrid XGBoost FAISS Model đã lưu vào 'GAT_XGB_Hybrid_FAISS_Mode

In [9]:
# thử GAT + XGB với CONCAT (Raw Features + Embeddings) để xem có cải thiện không
print("\n=== BƯỚC 4: TRÍCH XUẤT VÀ NỐI (CONCAT) FEATURES + EMBEDDINGS CHO HYBRID XGBOOST ===")
X_train_concat, y_train_concat = extract_concat_features_mask(model_faiss, super_graph_faiss, super_graph_faiss.train_mask, device=device)
X_valid_concat, y_valid_concat = extract_concat_features_mask(model_faiss, super_graph_faiss, super_graph_faiss.valid_mask, device=device)
X_test_concat, y_test_concat = extract_concat_features_mask(model_faiss, super_graph_faiss, super_graph_faiss.test_mask, device=device) 
hybrid_xgb_concat = train_evaluate_concat_xgboost(X_train_concat, y_train_concat, X_valid_concat, y_valid_concat, X_test_concat, y_test_concat)


=== BƯỚC 4: TRÍCH XUẤT VÀ NỐI (CONCAT) FEATURES + EMBEDDINGS CHO HYBRID XGBOOST ===
Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...


Đang trích xuất Vectơ Nối:   0%|          | 0/151 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Nối: (2470607, 378)
Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...


Đang trích xuất Vectơ Nối:   0%|          | 0/35 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Nối: (570114, 378)
Đang khởi tạo Inference Loader lấy Embeddings & Raw Features...


Đang trích xuất Vectơ Nối:   0%|          | 0/47 [00:00<?, ?it/s]

Trích xuất thành công! Kích thước Ma trận Nối: (760291, 378)

--- BẮT ĐẦU TRAIN XGBOOST TỪ FEATURE + EMBEDDING (CONCAT) ---
Đang tính toán Custom Smoothed Class Weights...
🧹 Đang dọn dẹp RAM & VRAM (Garbage Collection)...
⏳ Đang huấn luyện XGBoost... (Early Stopping trên Valid Set)
[0]	validation_0-mlogloss:1.56145	validation_1-mlogloss:1.56247
[100]	validation_0-mlogloss:0.00116	validation_1-mlogloss:0.01681
[141]	validation_0-mlogloss:0.00043	validation_1-mlogloss:0.01764
Huấn luyện xong XGBoost tự động dừng tại vòng tối ưu thứ: 101

--- ĐÁNH GIÁ (EVALUATION) TRÊN TẬP TEST MASK ---
Accuracy:            99.23%
F1-Score (Macro):    97.38%
F1-Score (Weighted): 99.23%

Classification Report:
              precision    recall  f1-score   support

           0     0.9725    0.9958    0.9840     19851
           1     0.9993    1.0000    0.9996    484073
           2     0.9960    0.9964    0.9962      2521
           3     0.9998    0.9957    0.9977     36260
           4     0.9663    0.9

In [10]:
# Dùng XGBoost Raw Features làm baseline để so sánh
print("\n=== BƯỚC 5: ĐÁNH GIÁ BASELINE XGBOOST CHỈ RAW FEATURES (KHÔNG DÙNG EMBEDDING) ===")
# Trích xuất Raw Features thuần túy cho từng Mask (Train/Valid/Test)
def extract_raw_features_mask(graph_data, mask):
    print("Đang trích xuất Raw Features thuần túy...")
    features = graph_data.x[mask].cpu().numpy()
    labels = graph_data.y[mask].cpu().numpy()
    print(f"Trích xuất thành công! Kích thước Ma trận Raw Features: {features.shape}")
    return features, labels
X_train_raw, y_train_raw = extract_raw_features_mask(super_graph_faiss, super_graph_faiss.train_mask)
X_valid_raw, y_valid_raw = extract_raw_features_mask(super_graph_faiss, super_graph_faiss.valid_mask)
X_test_raw, y_test_raw = extract_raw_features_mask(super_graph_faiss, super_graph_faiss.test_mask)
baseline_xgb_raw = train_evaluate_xgboost(X_train_raw, y_train_raw, X_valid_raw, y_valid_raw, X_test_raw, y_test_raw)



=== BƯỚC 5: ĐÁNH GIÁ BASELINE XGBOOST CHỈ RAW FEATURES (KHÔNG DÙNG EMBEDDING) ===
Đang trích xuất Raw Features thuần túy...
Trích xuất thành công! Kích thước Ma trận Raw Features: (2470607, 314)
Đang trích xuất Raw Features thuần túy...
Trích xuất thành công! Kích thước Ma trận Raw Features: (570114, 314)
Đang trích xuất Raw Features thuần túy...
Trích xuất thành công! Kích thước Ma trận Raw Features: (760291, 314)

--- BẮT ĐẦU TRAIN XGBOOST TỪ BASE GAT EMBEDDING ---
Đang tính toán Custom Smoothed Class Weights...
🧹 Đang dọn dẹp RAM & VRAM (Garbage Collection)...
⏳ Đang huấn luyện XGBoost... (Có sử dụng Valid Set để Tự động Dừng - Early Stopping)
[0]	validation_0-mlogloss:1.56558	validation_1-mlogloss:1.56566
[100]	validation_0-mlogloss:0.04259	validation_1-mlogloss:0.05382
[200]	validation_0-mlogloss:0.03143	validation_1-mlogloss:0.04573
[300]	validation_0-mlogloss:0.02720	validation_1-mlogloss:0.04276
[400]	validation_0-mlogloss:0.02484	validation_1-mlogloss:0.04178
[500]	validation

In [11]:
# đánh giá GAT trực tiếp trên tập Test Mask (Không qua XGBoost) để xem hiệu quả của GAT như một trình phân loại thuần túy
@torch.no_grad()
def evaluate_gat_directly(model, graph_data, mask, device='cpu'):
    model.eval()
    print("Đang khởi tạo NeighborLoader cho đánh giá trực tiếp GAT...")
    loader = NeighborLoader(
        graph_data,
        num_neighbors=[5, 3],
        input_nodes=mask,
        batch_size=16384,
        shuffle=False,
        num_workers=0 
    )
    all_preds = []
    all_labels = []
    pbar = tqdm(loader, desc="Đang đánh giá trực tiếp GAT")
    for batch in pbar:  
        batch = batch.to(device)
        out, _ = model(batch.x, batch.edge_index, batch.edge_attr)
        bs = batch.batch_size
        all_preds.append(out[:bs].cpu().numpy())
        all_labels.append(batch.y[:bs].cpu().numpy())
        del batch, out
    pbar.close()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    final_preds = np.concatenate(all_preds, axis=0)
    final_labels = np.concatenate(all_labels, axis=0)   
    pred_classes = final_preds.argmax(axis=1)
    acc = accuracy_score(final_labels, pred_classes)
    f1_macro = f1_score(final_labels, pred_classes, average='macro')
    f1_weighted = f1_score(final_labels, pred_classes, average='weighted')
    print(f"GAT Đánh giá trực tiếp trên Test Mask:")
    print(f"Accuracy:            {acc*100:.2f}%")
    print(f"F1-Score (Macro):    {f1_macro * 100:.2f}%")
    print(f"F1-Score (Weighted): {f1_weighted * 100:.2f}%\n")
    print("Classification Report:")
    print(classification_report(final_labels, pred_classes, digits=4))
evaluate_gat_directly(model_faiss, super_graph_faiss, super_graph_faiss.test_mask, device=device)

Đang khởi tạo NeighborLoader cho đánh giá trực tiếp GAT...


Đang đánh giá trực tiếp GAT:   0%|          | 0/47 [00:00<?, ?it/s]

GAT Đánh giá trực tiếp trên Test Mask:
Accuracy:            98.62%
F1-Score (Macro):    95.41%
F1-Score (Weighted): 98.64%

Classification Report:
              precision    recall  f1-score   support

           0     0.9659    0.8969    0.9301     19851
           1     0.9987    0.9979    0.9983    484073
           2     0.9347    1.0000    0.9663      2521
           3     0.9997    0.9900    0.9948     36260
           4     0.8194    0.9140    0.8641     18981
           5     0.9927    0.9963    0.9945      2460
           6     0.9420    0.8831    0.9116     11851
           7     1.0000    0.9930    0.9965    107440
           8     0.8394    0.9887    0.9080     16760
           9     0.9884    0.9250    0.9557     41522
          10     0.9519    0.9995    0.9751     18572

    accuracy                         0.9862    760291
   macro avg     0.9484    0.9622    0.9541    760291
weighted avg     0.9873    0.9862    0.9864    760291

